In [ ]:
# AGI Bench: Proactive/Retroactive Interference
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


# Proactive & Retroactive Interference Benchmark

**Cognitive Science**: Underwood (1957), Anderson (2003)

In [ ]:
"""
Novel Rule System Generator for Learning Benchmarks.

Generates procedural rule systems that cannot be in training data.
Each system defines a mapping from inputs to outputs via a chain
of deterministic rules. Difficulty is controlled by:
- Number of rules
- Number of input features
- Rule interaction complexity (independent vs. chained)

Systems are seeded for reproducibility across runs.
"""

import random
import hashlib
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]  # {"input": str, "output": str}
    test_items: list[dict]  # {"input": str, "output": str}
    difficulty: int  # 1-3
    n_rules: int
    domain: str  # "symbol", "language", "number"


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a symbol transformation rule system.

    Input: sequence of symbols (e.g., "△ ○ □")
    Rules: transformations (e.g., "△ followed by ○ becomes ★")
    Output: transformed sequence
    """
    rng = _make_rng(seed)

    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
    colors = ["red", "blue", "green", "yellow"]

    if difficulty == 1:
        # Simple 1-to-1 substitution
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        # Context-dependent: pairs matter
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:  # difficulty == 3
        # Multi-pass with conditional rules
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]
        cond = src[2]  # If this symbol is present, apply extra rule
        extra_map = {src[3]: dst[3]}

        rules = [
            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()
        ]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")

        def apply_rules(seq):
            # Pass 1
            result = [mapping1.get(s, s) for s in seq]
            # Pass 2
            result = [mapping2.get(s, s) for s in result]
            # Conditional
            if cond in seq:  # Check original sequence
                result = [extra_map.get(s, s) for s in result]
            return result

    # Generate examples
    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    # Deduplicate by input
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="symbol",
    )


def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a novel number system / arithmetic.

    Input: expression in the invented system
    Rules: how operators work
    Output: numeric result
    """
    rng = _make_rng(seed)

    op_names = ["grok", "flim", "zorp", "quex", "blix"]
    ops = rng.sample(op_names, 3)

    if difficulty == 1:
        # Two operators: basic arithmetic with twist
        a_op, b_op = ops[0], ops[1]
        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment
        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: absolute difference of x and y",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    elif difficulty == 2:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x * 2 + y
        b_fn = lambda x, y: (x + y) % 10
        c_fn = lambda x, y: max(x, y) - min(x, y) + 1
        rules = [
            f"'{a_op}(x, y)' means: double x, then add y",
            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",
            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",
        ]
        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}

    else:  # difficulty == 3
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        # Nested operations
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: x * y
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: multiply x and y",
            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    # Generate examples
    all_items = []
    for _ in range(20):
        if difficulty <= 2:
            op_name = rng.choice(list(op_map.keys()))
            x = rng.randint(1, 9)
            y = rng.randint(1, 9)
            result = op_map[op_name](x, y)
            expr = f"{op_name}({x}, {y})"
        else:
            # Allow nesting
            if rng.random() < 0.5:
                op_name = rng.choice(list(op_map.keys()))
                x = rng.randint(1, 9)
                y = rng.randint(1, 9)
                result = op_map[op_name](x, y)
                expr = f"{op_name}({x}, {y})"
            else:
                inner_op = rng.choice(list(op_map.keys()))
                outer_op = rng.choice(list(op_map.keys()))
                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
                inner_result = op_map[inner_op](x, y)
                result = op_map[outer_op](inner_result, z)
                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"

        all_items.append({"input": expr, "output": str(result)})

    # Deduplicate
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_ex = min(12, len(unique_items) - 5)
    examples = unique_items[:n_ex]
    test_items = unique_items[n_ex:n_ex + 5]

    return RuleSystem(
        name=f"NumberSystem-{seed}",
        description="Evaluate expressions using novel arithmetic operators",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="number",
    )


# Pre-generated systems for the benchmark
LEARNING_CURVE_SYSTEMS = [
    generate_symbol_system("lc_sym_easy", difficulty=1),
    generate_symbol_system("lc_sym_med", difficulty=2),
    generate_symbol_system("lc_sym_hard", difficulty=3),
    generate_number_system("lc_num_easy", difficulty=1),
    generate_number_system("lc_num_med", difficulty=2),
    generate_number_system("lc_num_hard", difficulty=3),
]

# Systems for transfer testing
TRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)
TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)
TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)

# Systems for interference testing
INTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)
INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)


In [ ]:
"""
Learning Benchmark 3: Proactive & Retroactive Interference

Tests whether learning new material interferes with previously
learned material (retroactive) and whether old knowledge impedes
new learning (proactive).

Cognitive Science Basis:
- Underwood (1957): Proactive inhibition in retention
- Postman (1961): Retroactive inhibition
- Anderson (2003): Retrieval-induced forgetting

Protocol:
1. Learn System A → Test A (baseline A)
2. Learn System B (similar to A) → Test B (baseline B)
3. Re-test A → Measure retroactive interference
4. Compare B learning rate vs. A (proactive interference)

Score: Measures resistance to interference (higher = better).
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import re
import json
# generate_symbol_system defined above


@dataclass
class InterfAnswer:
    answer: str


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    return e in m or m in e


def test_system(llm, system, context_prefix: str = "", chat_prefix: str = "test") -> float:
    """Test model on a system's test items. Returns accuracy."""
    correct = 0
    rules_text = f"**{system.name}**\n"
    for r in system.rules:
        rules_text += f"- {r}\n"
    examples_text = "\n**Examples:**\n"
    for ex in system.examples[:8]:
        examples_text += f"  {ex['input']} → {ex['output']}\n"

    for ti, test_item in enumerate(system.test_items):
        with kbench.chats.new(f"{chat_prefix}_{ti}"):
            prompt = (
                context_prefix +
                f"\nApply these rules:\n{rules_text}{examples_text}\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\"}}"
            )
            try:
                result = llm.prompt(prompt, schema=InterfAnswer)
                answer = result.answer
            except Exception:
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    answer = str(parsed.get("answer", raw))
                except Exception:
                    answer = raw

            if check_output(answer, test_item["output"]):
                correct += 1

    return correct / len(system.test_items) if system.test_items else 0


# Generate two similar systems that should interfere
SYSTEM_A = generate_symbol_system("interf_alpha_v2", difficulty=2)
SYSTEM_B = generate_symbol_system("interf_beta_v2", difficulty=2)


@kbench.task(name="learning_interference")
def learning_interference(llm) -> float:
    """
    Proactive & Retroactive Interference Benchmark.

    Measures how learning similar systems affects retention and acquisition.

    Protocol:
    1. Learn A → Test A (baseline_A)
    2. Learn B → Test B (baseline_B)
    3. After B: Re-test A (post_interference_A)
    4. Measure interference

    Score = 0.40 * (1 - retroactive_interference)
          + 0.30 * baseline_accuracy_A
          + 0.30 * baseline_accuracy_B

    Where retroactive_interference = max(0, baseline_A - post_interference_A)
    """

    # ── Phase 1: Learn System A, test A ──
    baseline_A = test_system(llm, SYSTEM_A, chat_prefix="phase1_A")

    # ── Phase 2: Learn System B, test B ──
    # Include A context to create potential interference
    a_context = (
        f"You previously learned system {SYSTEM_A.name}. "
        f"Now learn a NEW but similar system.\n"
    )
    baseline_B = test_system(llm, SYSTEM_B, context_prefix=a_context, chat_prefix="phase2_B")

    # ── Phase 3: Re-test A after learning B ──
    b_context = (
        f"You recently learned two similar systems. "
        f"Now I want you to recall the FIRST system ({SYSTEM_A.name}) specifically. "
        f"Ignore the second system you learned.\n"
    )
    post_interf_A = test_system(llm, SYSTEM_A, context_prefix=b_context, chat_prefix="phase3_retest_A")

    # ── Compute Metrics ──
    retroactive = max(0, baseline_A - post_interf_A)
    proactive = max(0, baseline_A - baseline_B)  # If B worse than A, proactive interference

    # Score: higher = better (less interference, higher accuracy)
    score = round(
        0.40 * (1 - retroactive)
        + 0.30 * baseline_A
        + 0.30 * baseline_B,
        4
    )

    # ── Logging ──
    print(f"\n{'='*60}")
    print(f"PROACTIVE & RETROACTIVE INTERFERENCE RESULTS")
    print(f"{'='*60}")
    print(f"System A: {SYSTEM_A.name} ({len(SYSTEM_A.rules)} rules)")
    print(f"System B: {SYSTEM_B.name} ({len(SYSTEM_B.rules)} rules)")
    print(f"\n--- Phase Results ---")
    print(f"Baseline A:          {baseline_A:.2%}")
    print(f"Baseline B:          {baseline_B:.2%}")
    print(f"Post-interference A: {post_interf_A:.2%}")
    print(f"\n--- Interference Metrics ---")
    print(f"Retroactive interference: {retroactive:.2%} (A drop after learning B)")
    print(f"Proactive interference:   {proactive:.2%} (B disadvantage vs A)")
    print(f"\nComposite score: {score:.4f}")

    return score


# ─── Run ────────────────────────────────────────────────────────────
learning_interference.run(llm=kbench.llm)
